In [2]:
import pandas as pd
import numpy as np
from scipy.signal import find_peaks

# --- CONFIGURATION ---
FILENAME = 'pulse_ox_1764746449.csv'

# Load the data
try:
    df = pd.read_csv(FILENAME)
except FileNotFoundError:
    print(f"Error: Could not find file {FILENAME}")
    exit()

# --- 1. DATA PRE-PROCESSING ---
# Filter out the startup saturation (values near 1023)
# We assume valid data is when the signal drops below 1020
valid_signal = df[(df['red'] < 1020) & (df['ir'] < 1020) & (df['red'] > 20)].copy()

if valid_signal.empty:
    print("Error: No valid data found (all data is saturated/railed). Check V_REF_VALUE.")
    exit()

# Extract arrays for processing
ir = valid_signal['ir'].values
red = valid_signal['red'].values
timestamps = valid_signal['timestamp'].values

# --- 2. SYSTEM METRICS (Timing) ---
# Calculate time differences between samples
dt = np.diff(timestamps)
mean_sampling_rate_ms = np.mean(dt)
sampling_frequency_hz = 1000.0 / mean_sampling_rate_ms
jitter_ms = np.std(dt)

# Settling Time: Time from start of recording until signal becomes valid
start_time = df['timestamp'].iloc[0]
lock_time = valid_signal['timestamp'].iloc[0]
settling_time_sec = (lock_time - start_time) / 1000.0

# --- 3. PHYSIOLOGICAL ACCURACY (BPM & SpO2) ---

# --- REVISED BPM LOGIC ---
# Dynamic prominence based on signal standard deviation
dynamic_prominence = np.std(ir) * 0.5 

# Find peaks
# Distance=12 samples (~60ms at 200Hz) prevents counting noise glitches
peaks, _ = find_peaks(ir, distance=12, prominence=dynamic_prominence)

bpm = 0
mean_ipi = 0
sdnn = 0

if len(peaks) > 1:
    peak_times = timestamps[peaks]
    intervals = np.diff(peak_times) 
    
    # Filter for biologically valid intervals (30 BPM to 200 BPM)
    valid_intervals = intervals[(intervals > 300) & (intervals < 2000)]
    
    if len(valid_intervals) > 0:
        mean_ipi = np.mean(valid_intervals)
        sdnn = np.std(valid_intervals) # HRV Metric
        bpm = 60000.0 / mean_ipi

# --- SpO2 CALCULATION ---
# DC = Mean, AC = Std Dev
red_dc = np.mean(red)
ir_dc = np.mean(ir)
red_ac = np.std(red)
ir_ac = np.std(ir)

spo2 = 0
ratio = 0
if red_dc > 0 and ir_dc > 0:
    ratio = (red_ac / red_dc) / (ir_ac / ir_dc)
    spo2 = 110 - (25 * ratio)

# --- 4. SIGNAL INTEGRITY (SNR Calculation) ---
def calculate_snr(data_array, window_size=10):
    # Smooth the signal to estimate the "ideal" wave
    series = pd.Series(data_array)
    signal_smooth = series.rolling(window=window_size, center=True).mean()
    
    # Isolate noise (Raw - Smooth)
    noise_residual = series - signal_smooth
    
    # Remove NaNs
    p_signal = np.var(signal_smooth.dropna())
    p_noise = np.var(noise_residual.dropna())
    
    if p_noise == 0: return 0, 0
    
    snr = 10 * np.log10(p_signal / p_noise)
    noise_rms = np.sqrt(p_noise)
    return snr, noise_rms

red_snr, red_noise = calculate_snr(red)
ir_snr, ir_noise = calculate_snr(ir)

# --- 5. REPORT GENERATION ---
print("="*40)
print(f"PULSE OXIMETER PERFORMANCE REPORT")
print(f"File: {FILENAME}")
print("="*40)

print(f"\n[1] SYSTEM LATENCY & TIMING")
print(f"Sampling Frequency:   {sampling_frequency_hz:.2f} Hz")
print(f"Sampling Jitter:      {jitter_ms:.3f} ms")
print(f"Lock-on/Settling Time:{settling_time_sec:.2f} s")

print(f"\n[2] SIGNAL INTEGRITY")
print(f"RED SNR:              {red_snr:.2f} dB")
print(f"IR SNR:               {ir_snr:.2f} dB")
print(f"Noise Floor (RMS):    {ir_noise:.2f} ADC units")
print(f"Dynamic Range Usage:  {(np.ptp(ir)/1023)*100:.1f}%")

print(f"\n[3] PHYSIOLOGICAL ACCURACY")
print(f"Calculated BPM:       {bpm:.1f}")
print(f"Heart Rate Var (SDNN):{sdnn:.1f} ms")
print(f"Calculated SpO2:      {spo2:.1f}%")
print(f"Ratio of Ratios (R):  {ratio:.3f}")

print("\n[4] INTERPRETATION")
if bpm > 50 and bpm < 120:
    print(" > BPM is within normal resting range.")
else:
    print(" > BPM is outside normal range (check for motion artifacts).")

if ir_snr > 15:
    print(" > SNR is Excellent (>15dB). Analog filtering is effective.")
else:
    print(" > SNR is Low (<15dB). Signal is noisy.")
print("="*40)

PULSE OXIMETER PERFORMANCE REPORT
File: pulse_ox_1764746449.csv

[1] SYSTEM LATENCY & TIMING
Sampling Frequency:   474.75 Hz
Sampling Jitter:      408.892 ms
Lock-on/Settling Time:2.25 s

[2] SIGNAL INTEGRITY
RED SNR:              16.59 dB
IR SNR:               20.74 dB
Noise Floor (RMS):    9.24 ADC units
Dynamic Range Usage:  59.1%

[3] PHYSIOLOGICAL ACCURACY
Calculated BPM:       70.1
Heart Rate Var (SDNN):175.1 ms
Calculated SpO2:      92.0%
Ratio of Ratios (R):  0.721

[4] INTERPRETATION
 > BPM is within normal resting range.
 > SNR is Excellent (>15dB). Analog filtering is effective.
